# 04 Green Time Allocation

**Description:** Allocate signal durations from lane density scores.

**Objective:** Build a simple adaptive schedule and save the results to `outputs/green_time/`.

## Step 1. Setup

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Outputs root: {OUTPUT_ROOT}')

OUTPUT_DIR = OUTPUT_ROOT / 'green_time'
os.makedirs(OUTPUT_DIR, exist_ok=True)
INPUT_CSV = OUTPUT_ROOT / 'density_scoring' / 'density_scores.csv'
SCHEDULE_CSV = OUTPUT_DIR / 'green_time_schedule.csv'
SCHEDULE_PLOT = OUTPUT_DIR / 'green_time_schedule.png'


In [ ]:
if INPUT_CSV.exists():
    density_df = pd.read_csv(INPUT_CSV)
else:
    density_df = pd.DataFrame({
        'lane_id': ['lane_1', 'lane_2', 'lane_3', 'lane_4'],
        'vehicle_count': [3, 7, 2, 1],
        'density_score': [0.15, 0.35, 0.10, 0.05],
    })

density_df


## Step 2. Allocation Logic

This skeleton uses proportional allocation with a minimum green time to keep the example stable.

In [ ]:
def allocate_green_time(df: pd.DataFrame, cycle_time: int = 120, minimum_green: int = 10) -> pd.DataFrame:
    schedule = df.copy()
    total_density = schedule['density_score'].sum()
    if total_density <= 0:
        schedule['green_time_sec'] = minimum_green
    else:
        schedule['green_time_sec'] = schedule['density_score'].apply(
            lambda score: max(minimum_green, round((score / total_density) * cycle_time))
        )
    return schedule

schedule_df = allocate_green_time(density_df)
schedule_df.to_csv(SCHEDULE_CSV, index=False)
print(f'Saved schedule to {SCHEDULE_CSV}')
schedule_df


## Step 3. Save Visualization

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(schedule_df['lane_id'], schedule_df['green_time_sec'], color='seagreen')
plt.title('Allocated Green Time by Lane')
plt.xlabel('Lane')
plt.ylabel('Green Time (sec)')
plt.tight_layout()
plt.savefig(SCHEDULE_PLOT, dpi=150)
plt.show()
print(f'Saved plot to {SCHEDULE_PLOT}')
